# ML Enhancement: Stockout Prediction

Predicts stockout risk using a **distributed Spark ML `GBTClassifier`**
with **MLflow** experiment tracking.  All computation stays in Spark — no
`toPandas()` or `.collect()` on large data.

## Business Value
- Proactive replenishment before stockouts occur
- Reduce lost sales from out-of-stock items
- Optimize safety stock levels
- Prioritize expedited shipping decisions

## Model Details
- **Algorithm:** Spark ML `GBTClassifier` (distributed, Fabric-native)
- **Target:** Binary stockout within a configurable forecast horizon
- **Features:** Current inventory, demand velocity, time features, product dimensions
- **Tracking:** Fabric-native MLflow (parameters, metrics, model artifact)
- **Performance Target:** Recall > 0.8, Precision > 0.5

## Data Flow
```
Silver (fact_store_inventory_txn, fact_receipt_lines, fact_receipts, dim_products)
    --> Feature Engineering (Spark)
    --> VectorAssembler
    --> GBTClassifier (distributed)
    --> BinaryClassificationEvaluator
    --> gold_stockout_risk (Gold Delta table)
    --> MLflow Experiment
```

## Schedule
Run this notebook on a configurable cadence (daily is a common default)
to generate updated stockout risk scores.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.utils import AnalysisException
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.functions import vector_to_array
from datetime import datetime, timedelta, timezone
import mlflow
import os


In [ ]:
# =============================================================================
# PARAMETERS
# =============================================================================

def get_env(var_name, default=None):
    return os.environ.get(var_name, default)

SILVER_DB = get_env("SILVER_DB", default="ag")
GOLD_DB = get_env("GOLD_DB", default="au")
EXPERIMENT_NAME = get_env("MLFLOW_EXPERIMENT", default="stockout_prediction")
INVENTORY_TXN_TABLE = get_env("INVENTORY_TXN_TABLE", default="fact_store_inventory_txn")
RECEIPT_LINES_TABLE = get_env("RECEIPT_LINES_TABLE", default="fact_receipt_lines")
RECEIPTS_TABLE = get_env("RECEIPTS_TABLE", default="fact_receipts")
PRODUCTS_TABLE = get_env("PRODUCTS_TABLE", default="dim_products")
STOCKOUT_RISK_TABLE = get_env("STOCKOUT_RISK_TABLE", default="gold_stockout_risk")
STOCKOUT_RISK_TABLE_NAME = f"{GOLD_DB}.{STOCKOUT_RISK_TABLE}"
SCHEDULE_CADENCE = get_env("SCHEDULE_CADENCE", default="daily")
LABEL_COL = "stockout_label"

# Model parameters
FORECAST_HORIZON_DAYS = int(get_env("FORECAST_HORIZON_DAYS", default="3"))
LOOKBACK_DAYS = int(get_env("LOOKBACK_DAYS", default="30"))
STOCKOUT_THRESHOLD = float(get_env("STOCKOUT_THRESHOLD", default="0"))

# GBT hyperparameters
MAX_ITER = int(get_env("MAX_ITER", default="80"))
MAX_DEPTH = int(get_env("MAX_DEPTH", default="5"))
STEP_SIZE = float(get_env("STEP_SIZE", default="0.1"))
SUBSAMPLING_RATE = float(get_env("SUBSAMPLING_RATE", default="1.0"))

print(f"Configuration: SILVER_DB={SILVER_DB}, GOLD_DB={GOLD_DB}")
print(f"Source tables: {SILVER_DB}.{INVENTORY_TXN_TABLE}, {SILVER_DB}.{RECEIPT_LINES_TABLE}, {SILVER_DB}.{RECEIPTS_TABLE}, {SILVER_DB}.{PRODUCTS_TABLE}")
print(f"Output table: {STOCKOUT_RISK_TABLE_NAME}")
print(f"MLflow experiment: {EXPERIMENT_NAME}")
print(f"Forecast horizon: {FORECAST_HORIZON_DAYS} days")
print(f"Lookback period: {LOOKBACK_DAYS} days")
print(f"Schedule cadence: {SCHEDULE_CADENCE}")
print(f"GBTClassifier: maxIter={MAX_ITER}, maxDepth={MAX_DEPTH}, stepSize={STEP_SIZE}, subsamplingRate={SUBSAMPLING_RATE}")


In [ ]:
# =============================================================================
# HELPER FUNCTIONS & MLFLOW SETUP
# =============================================================================

def ensure_database(name):
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {name}")

def read_silver(table_name):
    return spark.table(f"{SILVER_DB}.{table_name}")

def silver_exists(table_name):
    try:
        spark.table(f"{SILVER_DB}.{table_name}")
        return True
    except AnalysisException:
        return False

ensure_database(GOLD_DB)

# Fabric auto-configures the MLflow tracking URI
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"MLflow experiment '{EXPERIMENT_NAME}' ready")


## Step 1: Calculate Current Inventory Position and Historical Demand


In [ ]:
# Validate required Silver tables
required_tables = [INVENTORY_TXN_TABLE, RECEIPT_LINES_TABLE, RECEIPTS_TABLE, PRODUCTS_TABLE]
for tbl in required_tables:
    if not silver_exists(tbl):
        raise RuntimeError(f"{tbl} table not found in Silver layer")

print("=" * 80)
print("EXTRACTING INVENTORY AND DEMAND DATA")
print("=" * 80)
print()

# Get current inventory position per store/product
print("Loading current inventory position...")
inventory_df = (
    read_silver(INVENTORY_TXN_TABLE)
    .select("store_id", "product_id", "event_ts", "balance", "delta")
)

# Get latest inventory snapshot per store/product
window_latest = Window.partitionBy("store_id", "product_id").orderBy(F.desc("event_ts"))
current_inventory = (
    inventory_df
    .withColumn("rn", F.row_number().over(window_latest))
    .filter(F.col("rn") == 1)
    .select(
        "store_id",
        "product_id",
        F.col("event_ts").alias("inventory_as_of"),
        F.col("balance").alias("current_inventory")
    )
)

print(f"  Current inventory records: {current_inventory.count()}")
print()

# Calculate demand velocity from sales (last 7, 14, 30 days)
print("Calculating demand velocity from sales...")
sales_df = (
    read_silver(RECEIPT_LINES_TABLE)
    .join(
        read_silver(RECEIPTS_TABLE).select("receipt_id_ext", "store_id", "event_ts"),
        on="receipt_id_ext",
        how="inner"
    )
    .select("store_id", "product_id", "quantity", "event_ts")
    .filter(F.col("event_ts") >= F.current_timestamp() - F.expr(f"INTERVAL {LOOKBACK_DAYS} DAYS"))
)

# Calculate demand for different time windows
sales_df = sales_df.withColumn("days_ago", F.datediff(F.current_timestamp(), F.col("event_ts")))

demand_velocity = (
    sales_df
    .groupBy("store_id", "product_id")
    .agg(
        F.sum(F.when(F.col("days_ago") <= 7, F.col("quantity")).otherwise(0)).alias("demand_7d"),
        F.sum(F.when(F.col("days_ago") <= 14, F.col("quantity")).otherwise(0)).alias("demand_14d"),
        F.sum(F.when(F.col("days_ago") <= 30, F.col("quantity")).otherwise(0)).alias("demand_30d"),
        F.avg(F.when(F.col("days_ago") <= 30, F.col("quantity")).otherwise(None)).alias("avg_order_size_30d")
    )
    .withColumn("demand_velocity_daily", F.col("demand_30d") / 30.0)
)

print(f"  Demand velocity records: {demand_velocity.count()}")
print()


## Step 2: Feature Engineering


In [ ]:
print("=" * 80)
print("FEATURE ENGINEERING")
print("=" * 80)
print()

# Join inventory with demand
features_df = (
    current_inventory
    .join(demand_velocity, on=["store_id", "product_id"], how="left")
)

# Load product dimension for department/category
print("Loading product attributes...")
products_df = (
    read_silver(PRODUCTS_TABLE)
    .select("ID", "Department", "Category", "Subcategory")
    .withColumnRenamed("ID", "product_id")
)

features_df = features_df.join(products_df, on="product_id", how="left")

# Add time-based features
print("Adding time-based features...")
features_df = (
    features_df
    .withColumn("day_of_week", F.dayofweek(F.col("inventory_as_of")))
    .withColumn("day_of_month", F.dayofmonth(F.col("inventory_as_of")))
    .withColumn("week_of_year", F.weekofyear(F.col("inventory_as_of")))
    .withColumn("is_weekend", F.when(F.dayofweek(F.col("inventory_as_of")).isin([1, 7]), 1).otherwise(0))
)

# Calculate derived features
print("Calculating derived features...")
features_df = (
    features_df
    # Fill nulls in demand metrics with 0
    .fillna(0, subset=["demand_7d", "demand_14d", "demand_30d", "demand_velocity_daily", "avg_order_size_30d"])
    # Days of inventory remaining (current inventory / daily demand)
    .withColumn(
        "days_of_inventory",
        F.when(F.col("demand_velocity_daily") > 0, F.col("current_inventory") / F.col("demand_velocity_daily")).otherwise(999)
    )
    # Demand trend (comparing 7d vs 30d normalized demand)
    .withColumn(
        "demand_trend",
        F.when(
            F.col("demand_30d") > 0,
            (F.col("demand_7d") / 7.0) / (F.col("demand_30d") / 30.0)
        ).otherwise(1.0)
    )
    # Inventory ratio
    .withColumn(
        "inventory_ratio",
        F.when(F.col("demand_30d") > 0, F.col("current_inventory") / F.col("demand_30d")).otherwise(1.0)
    )
)

print(f"  Total feature records: {features_df.count()}")
print()

# Show sample features
print("Sample features:")
features_df.select(
    "store_id", "product_id", "current_inventory", "demand_velocity_daily",
    "days_of_inventory", "demand_trend", "day_of_week"
).show(10)


## Step 3: Create Training Labels (Historical Stockouts)


In [ ]:
print("=" * 80)
print("CREATING TRAINING LABELS")
print("=" * 80)
print()

# Find historical stockout events
print(f"Identifying stockouts (balance <= {STOCKOUT_THRESHOLD})...")

# Get all inventory transactions and identify stockouts
inventory_history = (
    read_silver(INVENTORY_TXN_TABLE)
    .select("store_id", "product_id", "event_ts", "balance")
    .withColumn("is_stockout", F.when(F.col("balance") <= STOCKOUT_THRESHOLD, 1).otherwise(0))
)

# For each point in time, look forward N days to see if stockout occurred
window_forward = Window.partitionBy("store_id", "product_id").orderBy("event_ts")

labeled_data = (
    inventory_history
    .withColumn("next_event_ts", F.lead("event_ts", 1).over(window_forward))
    .withColumn("next_balance", F.lead("balance", 1).over(window_forward))
    .withColumn(
        "days_to_next_event",
        F.when(
            F.col("next_event_ts").isNotNull(),
            F.datediff(F.col("next_event_ts"), F.col("event_ts"))
        ).otherwise(999)
    )
    .withColumn(
        LABEL_COL,
        F.when(
            (F.col("days_to_next_event") <= FORECAST_HORIZON_DAYS) & (F.col("next_balance") <= STOCKOUT_THRESHOLD),
            1
        ).otherwise(0)
    )
)

# Join labels with features based on store/product
training_data = (
    labeled_data
    .select("store_id", "product_id", "event_ts", "balance", LABEL_COL)
    .join(
        features_df,
        on=["store_id", "product_id"],
        how="inner"
    )
    .filter(F.col("event_ts") <= F.current_timestamp() - F.expr(f"INTERVAL {FORECAST_HORIZON_DAYS} DAYS"))
)

print(f"  Training records: {training_data.count()}")
print()

# Check class distribution
print("Class distribution:")
training_data.groupBy(LABEL_COL).count().show()


## Step 4: Train GBTClassifier Model (Distributed)

Uses Spark ML `GBTClassifier` — training runs distributed across Spark
executors.  All data stays in Spark DataFrames (no `toPandas()`).


In [ ]:
print("=" * 80)
print("TRAINING GBTCLASSIFIER MODEL (DISTRIBUTED)")
print("=" * 80)
print()

# Define numeric feature columns
feature_cols = [
    "current_inventory",
    "demand_7d",
    "demand_14d",
    "demand_30d",
    "demand_velocity_daily",
    "avg_order_size_30d",
    "days_of_inventory",
    "demand_trend",
    "inventory_ratio",
    "day_of_week",
    "day_of_month",
    "week_of_year",
    "is_weekend",
]

# Cast feature columns to double and fill nulls
for col_name in feature_cols:
    training_data = training_data.withColumn(col_name, F.col(col_name).cast("double"))
training_data = training_data.na.fill(0.0, subset=feature_cols)

# Cast label to double for Spark ML
training_data = training_data.withColumn(LABEL_COL, F.col(LABEL_COL).cast("double"))

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep",
)
df_assembled = assembler.transform(training_data)

# Train/test split (80/20)
df_train, df_test = df_assembled.randomSplit([0.8, 0.2], seed=42)

train_count = df_train.count()
test_count = df_test.count()
print(f"  Training set: {train_count} rows")
print(f"  Test set:     {test_count} rows")
print()

# Configure Spark ML GBT classifier
gbt = GBTClassifier(
    featuresCol="features",
    labelCol=LABEL_COL,
    predictionCol="stockout_predicted",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=MAX_ITER,
    maxDepth=MAX_DEPTH,
    stepSize=STEP_SIZE,
    subsamplingRate=SUBSAMPLING_RATE,
    seed=42,
)

print("Training GBTClassifier (distributed across Spark executors)...")

with mlflow.start_run(
    run_name=f"gbt_stockout_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M')}"
) as run:
    model = gbt.fit(df_train.select("features", LABEL_COL))

    # ── Evaluate on held-out test set ──────────────────────────────────
    df_preds = model.transform(df_test)

    # Extract stockout probability from probability vector (index 1 = positive class)
    df_preds = df_preds.withColumn(
        "stockout_probability", vector_to_array("probability")[1]
    )

    # AUC-ROC via BinaryClassificationEvaluator
    binary_eval = BinaryClassificationEvaluator(
        rawPredictionCol="rawPrediction",
        labelCol=LABEL_COL,
        metricName="areaUnderROC",
    )
    auc_roc = binary_eval.evaluate(df_preds)

    # Precision, Recall, F1 via MulticlassClassificationEvaluator
    mc_eval = MulticlassClassificationEvaluator(
        predictionCol="stockout_predicted",
        labelCol=LABEL_COL,
    )
    precision = mc_eval.evaluate(df_preds, {mc_eval.metricName: "weightedPrecision"})
    recall = mc_eval.evaluate(df_preds, {mc_eval.metricName: "weightedRecall"})
    f1 = mc_eval.evaluate(df_preds, {mc_eval.metricName: "f1"})

    # Log parameters
    mlflow.log_params({
        "algorithm": "gbt_classifier",
        "forecast_horizon_days": FORECAST_HORIZON_DAYS,
        "lookback_days": LOOKBACK_DAYS,
        "max_iter": MAX_ITER,
        "max_depth": MAX_DEPTH,
        "step_size": STEP_SIZE,
        "subsampling_rate": SUBSAMPLING_RATE,
        "feature_count": len(feature_cols),
        "train_rows": train_count,
        "test_rows": test_count,
    })

    # Log metrics
    mlflow.log_metrics({
        "auc_roc": round(auc_roc, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
    })

    # Log model artifact
    mlflow.spark.log_model(model, "gbt_stockout_model")

    print(f"MLflow run: {run.info.run_id}")
    print()
    print(f"Test-set evaluation:")
    print(f"  AUC-ROC:   {auc_roc:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print()

    # Acceptance criteria check
    print("=" * 80)
    print("ACCEPTANCE CRITERIA CHECK")
    print("=" * 80)
    print(f"Recall > 0.8:    {'PASS' if recall > 0.8 else 'FAIL'} (actual: {recall:.3f})")
    print(f"Precision > 0.5: {'PASS' if precision > 0.5 else 'FAIL'} (actual: {precision:.3f})")
    print()

    if recall <= 0.8 or precision <= 0.5:
        print("Below target - consider tuning hyperparameters or adding features")


## Step 5: Generate Predictions for Current Inventory

Score the full current-inventory feature set using the trained model.
All prediction stays in Spark — no `toPandas()`.


In [ ]:
print("=" * 80)
print("GENERATING STOCKOUT RISK PREDICTIONS")
print("=" * 80)
print()

# Prepare current features for scoring
df_current = features_df
for col_name in feature_cols:
    df_current = df_current.withColumn(col_name, F.col(col_name).cast("double"))
df_current = df_current.na.fill(0.0, subset=feature_cols)

# Assemble feature vector for scoring
df_current_assembled = assembler.transform(df_current)

# Score with trained model
print("Scoring current inventory with trained model...")
df_scored = model.transform(df_current_assembled)

# Extract stockout probability from probability vector
df_scored = df_scored.withColumn(
    "stockout_probability", vector_to_array("probability")[1]
)

# Add risk level categories
df_scored = df_scored.withColumn(
    "risk_level",
    F.when(F.col("stockout_probability") >= 0.7, "HIGH")
     .when(F.col("stockout_probability") >= 0.4, "MEDIUM")
     .otherwise("LOW")
)
df_scored = df_scored.withColumn("risk", F.col("risk_level"))
risk_rank_window = Window.orderBy(F.desc("stockout_probability"), F.asc("store_id"), F.asc("product_id"))
df_scored = df_scored.withColumn("ranking", F.row_number().over(risk_rank_window))

# Add metadata columns
df_scored = (
    df_scored
    .withColumn("predicted_at", F.lit(datetime.now(timezone.utc)))
    .withColumn("forecast_horizon_days", F.lit(FORECAST_HORIZON_DAYS))
)

print(f"  Predictions generated: {df_scored.count()}")
print()

# Show risk distribution
print("Risk Distribution:")
df_scored.groupBy("risk_level").count().orderBy("risk_level").show()

# Show top risks
print("Top 20 Stockout Risks:")
df_scored.select(
    "store_id", "product_id", "current_inventory", "demand_velocity_daily",
    "days_of_inventory", "stockout_probability", "risk_level"
).orderBy(F.desc("stockout_probability")).show(20)


## Step 6: Save to Gold Layer


In [ ]:
print("=" * 80)
print("SAVING TO GOLD LAYER")
print("=" * 80)
print()

# Select and cast output columns
df_output = (
    df_scored
    .select(
        F.col("store_id").cast("long"),
        F.col("product_id").cast("long"),
        F.col("current_inventory").cast("long"),
        F.col("demand_velocity_daily").cast("double"),
        F.col("days_of_inventory").cast("double"),
        F.col("demand_trend").cast("double"),
        F.col("stockout_probability").cast("double"),
        F.col("stockout_predicted").cast("int"),
        F.col("risk").cast("string"),
        F.col("ranking").cast("int"),
        F.col("risk_level").cast("string"),
        F.col("predicted_at").cast("timestamp"),
        F.col("forecast_horizon_days").cast("int"),
        F.col("Department").cast("string"),
        F.col("Category").cast("string"),
        F.col("Subcategory").cast("string"),
    )
)

# Save to Gold Delta table
table_name = STOCKOUT_RISK_TABLE_NAME
print(f"Saving to {table_name}...")
df_output.write.format("delta").mode("overwrite").saveAsTable(table_name)

row_count = spark.table(table_name).count()
print(f"  {table_name}: {row_count} rows")
print()


In [ ]:
print("=" * 80)
print("STOCKOUT PREDICTION COMPLETE")
print("=" * 80)
print()

gold_tables = spark.sql(f"SHOW TABLES IN {GOLD_DB}").collect()
print(f"Gold layer ({GOLD_DB}): {len(gold_tables)} tables")
print(f"Output table: {STOCKOUT_RISK_TABLE_NAME}")
print(f"Forecast horizon: {FORECAST_HORIZON_DAYS} days")
print()
print(f"MLflow experiment: {EXPERIMENT_NAME}")
print("Model logged: gbt_stockout_model")
print("View results in Fabric > Experiments")
print()
print("Next Steps:")
print("  1. Create Power BI dashboard for replenishment alerts")
print("  2. Set up notifications for HIGH risk items")
print(f"  3. Schedule this notebook to run on a {SCHEDULE_CADENCE} cadence")
print("  4. Monitor prediction accuracy and retrain as needed")
